# 05 — Supervised Fine-Tuning with Quantized Low-Rank Adaptation

## Objective
Prepare instruction-chat training records from the PubMedQA training partition, inspect completion-only masking, fit a Low-Rank Adaptation (LoRA) adapter with optional four-bit base-model quantization, and save reproducible checkpoint metadata.

## Research design
- **Source:** PubMedQA `pqa_labeled`, only the publication-aware training partition established in notebook 03.
- **Model:** same `MODEL_ID` used by the starting-model evaluation. Changing the model ID invalidates direct checkpoint comparisons.
- **Training target:** the assistant completion (`yes`, `no`, `maybe`), not the entire biomedical question or abstract. This avoids learning to copy prompts.
- **Training budget:** deliberately small for a local GPU demonstration; learning curves alone are not evidence of generalization.
- **Artifacts:** adapter, tokenizer, run metadata, and the trainer's available log history.

**Terminology:** SFT (Supervised Fine-Tuning); QLoRA (Quantized Low-Rank Adaptation); VRAM (Video Random Access Memory).

### Workspace and source-data manifest

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


In [2]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Training-compatible messages
The prompt uses the same system and user messages as the baseline in notebook 04. A standard prompt-completion dataset allows Transformer Reinforcement Learning (TRL) to mask prompt tokens when computing loss.

In [3]:
SYSTEM_INSTRUCTION = (
    "Answer the biomedical question using ONLY the supplied abstract evidence. "
    "Choose exactly one decision label: yes, no, or maybe. "
    "Return the label alone, without an explanation. "
    "Use maybe when the evidence does not establish yes or no."
)


def question_messages(row):
    """Use one consistent role-aware input protocol across model variants."""
    full_text = evidence_text(row)
    if not full_text:
        raise ValueError("Question has no evidence passage.")
    # Keep both the abstract introduction and conclusion; the conclusion often
    # carries the decision evidence. Apply the same limit in training/evaluation.
    if len(full_text) > 1600:
        text = full_text[:950] + "\n[Middle of abstract omitted for length]\n" + full_text[-600:]
    else:
        text = full_text
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {row['question']}\n\nAbstract evidence:\n{text}"},
    ]


def parse_decision(generated_text):
    """Parse an exact one-label answer; preserve raw outputs when parsing fails."""
    import re
    text = str(generated_text).strip()
    match = re.fullmatch(r"(?i)(yes|no|maybe)[.\s]*", text)
    return match.group(1).lower() if match else "invalid"

### Inspect one synthetic training record before using public data
This example checks role ordering and label placement; it is not biomedical training.

In [4]:
illustrative = {
    "pubid": "synthetic-example",
    "question": "Does an observed association prove causation?",
    "context": {"contexts": ["A correlation was observed in a non-randomized sample."]},
    "final_decision": "no",
}
prompt = question_messages(illustrative)
completion = [{"role": "assistant", "content": illustrative["final_decision"]}]
print("Message roles:", [message["role"] for message in prompt + completion])
print("Target completion:", completion[0]["content"])
assert parse_decision(completion[0]["content"]) == "no"

Message roles: ['system', 'user', 'assistant']
Target completion: no


### Supervised fine-tuning experiment

Set `RUN_TRAINING=1` after verifying the notebook 03 manifest. The default uses an unquantized bfloat16 base with LoRA because the installed PyTorch/Transformers combination cannot load this model in four bits. `SFT_USE_4BIT=1` requests QLoRA explicitly and will stop if that path is incompatible; it will not silently switch methods. Both paths mask every prompt token and optimize only the assistant completion. The run manifest records which method actually ran.

In [5]:
import gc
import json
import os
import time

model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
checkpoint_dir = RUNS / "sft_adapter"
print("Starting checkpoint:", model_id)
print("Adapter output:", checkpoint_dir)

if os.getenv("RUN_TRAINING", "0") == "1":
    import numpy as np
    import torch
    import transformers
    from datasets import Dataset, config, load_dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
        Trainer, TrainingArguments, default_data_collator,
    )

    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cache_root = Path(config.HF_DATASETS_CACHE)
        cached_files = list(cache_root.glob("qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
        if len(cached_files) != 1:
            raise FileNotFoundError(f"Expected one cached PubMedQA labeled Arrow file; found {len(cached_files)}")
        raw = Dataset.from_file(str(cached_files[0]))
        print("Source mode: existing Hugging Face Arrow cache")
    else:
        raw = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
        print("Source mode: Hugging Face load_dataset")

    partitions = prepare_partitions(raw, create_manifest=False)
    records = sorted(partitions["train"], key=paper_id)
    limit = int(os.getenv("SFT_TRAIN_MAX_ITEMS", "200"))
    if limit > 0 and limit < len(records):
        selected = np.random.default_rng(SEED).choice(len(records), size=limit, replace=False)
        records = [records[int(i)] for i in sorted(selected)]
    selected_ids = [paper_id(row) for row in records]
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    if not set(selected_ids).issubset(set(manifest["ids"]["train"])):
        raise ValueError("Training selection contains IDs outside the frozen training partition.")
    print("Training sample count:", len(records))
    print("Label distribution:", dict(Counter(str(row["final_decision"]).lower() for row in records)))

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.chat_template is None:
        raise ValueError("The selected tokenizer has no chat template.")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    max_length = 768
    def encode_training_record(row):
        prompt_messages = question_messages(row)
        completion_messages = [{"role": "assistant", "content": str(row["final_decision"]).lower()}]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        full_text = tokenizer.apply_chat_template(
            prompt_messages + completion_messages, tokenize=False, add_generation_prompt=False
        )
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
        full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(f"Chat-template token prefix mismatch for paper {paper_id(row)}")
        if len(full_ids) > max_length:
            raise ValueError(f"Training example {paper_id(row)} exceeds {max_length} tokens")
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        if not any(token != -100 for token in labels):
            raise ValueError(f"No assistant target tokens for paper {paper_id(row)}")
        return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

    encoded_rows = [encode_training_record(row) for row in records]
    train_data = Dataset.from_list(encoded_rows)
    prompt_tokens_masked = sum(sum(token == -100 for token in row["labels"]) for row in encoded_rows)
    target_tokens = sum(sum(token != -100 for token in row["labels"]) for row in encoded_rows)
    print("Prompt tokens masked:", prompt_tokens_masked, "Assistant target tokens:", target_tokens)
    print("Encoded length range:", min(map(lambda row: len(row["input_ids"]), encoded_rows)),
          max(map(lambda row: len(row["input_ids"]), encoded_rows)))

    use_cuda = torch.cuda.is_available()
    use_4bit = os.getenv("SFT_USE_4BIT", "0") == "1"
    if use_4bit and not use_cuda:
        raise RuntimeError("Four-bit training requires CUDA.")
    compute_dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    quant_config = (BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute_dtype
    ) if use_4bit else None)
    torch.manual_seed(SEED)
    if use_cuda:
        torch.cuda.manual_seed_all(SEED)
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto" if use_cuda else None,
        torch_dtype=compute_dtype if use_cuda and not use_4bit else None,
    )
    model.config.use_cache = False
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    adapter = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, adapter)
    model.print_trainable_parameters()

    settings = TrainingArguments(
        output_dir=str(checkpoint_dir),
        max_steps=int(os.getenv("SFT_MAX_STEPS", "40")),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=5,
        report_to="none",
        save_strategy="no",
        bf16=use_cuda and compute_dtype == torch.bfloat16,
        fp16=use_cuda and compute_dtype == torch.float16,
        seed=SEED,
        optim="adamw_torch",
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model, args=settings, train_dataset=train_data,
        data_collator=default_data_collator,
    )
    result = trainer.train()
    elapsed_seconds = time.perf_counter() - start
    peak_gpu_gib = torch.cuda.max_memory_allocated() / 2**30 if use_cuda else None
    trainer.save_model(str(checkpoint_dir))
    tokenizer.save_pretrained(str(checkpoint_dir))

    training_record = {
        "variant": "sft", "method": "QLoRA-4bit" if use_4bit else "LoRA-unquantized",
        "model_id": model_id,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "dataset_fingerprint": manifest["fingerprint"],
        "source_partition": "train", "selected_ids": selected_ids,
        "n_train": len(records), "seed": SEED,
        "max_steps": settings.max_steps, "completed_steps": trainer.state.global_step,
        "max_length": max_length, "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4, "learning_rate": 2e-4,
        "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
        "lora_target_modules": ["q_proj", "v_proj"],
        "prompt_tokens_masked": prompt_tokens_masked,
        "assistant_target_tokens": target_tokens,
        "training_loss": result.training_loss,
        "elapsed_seconds": round(elapsed_seconds, 3),
        "peak_gpu_allocated_gib": round(peak_gpu_gib, 3) if peak_gpu_gib is not None else None,
        "torch_version": torch.__version__, "transformers_version": transformers.__version__,
        "run_status": "trainer.train() completed",
    }
    (checkpoint_dir / "run_manifest.json").write_text(
        json.dumps(training_record, indent=2) + "\n", encoding="utf-8"
    )
    (checkpoint_dir / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=2) + "\n", encoding="utf-8"
    )
    print("Completed", training_record["method"], "SFT in", round(elapsed_seconds, 3),
          "seconds; training loss:", round(result.training_loss, 4),
          "peak GPU allocated GiB:", training_record["peak_gpu_allocated_gib"])
    del trainer, model
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
else:
    print("Training not enabled; set RUN_TRAINING=1 after reviewing memory requirements.")

Starting checkpoint: Qwen/Qwen2.5-0.5B-Instruct
Adapter output: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\sft_adapter
Source mode: existing Hugging Face Arrow cache
train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}
Training sample count: 200
Label distribution: {'yes': 110, 'maybe': 27, 'no': 63}


`torch_dtype` is deprecated! Use `dtype` instead!


Prompt tokens masked: 78535 Assistant target tokens: 600
Encoded length range: 210 627


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
5,0.601909
10,0.365594
15,0.492857
20,0.348961
25,0.352975
30,0.182965
35,0.243658
40,0.168964


Completed LoRA-unquantized SFT in 13.933 seconds; training loss: 0.3447 peak GPU allocated GiB: 2.627


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in Qwen/Qwen2.5-0.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


### Results

The run completed 40 optimizer steps on 200 PubMedQA training IDs with an unquantized bfloat16 base and LoRA. It masked 78,535 prompt tokens, optimized 600 assistant-target tokens, and logged mean training loss 0.3447. The 540,672 trainable adapter parameters were 0.1093% of the reported model parameter count.

### Interpretation

The saved adapter and manifest establish a completed, traceable SFT run. The executed method was not four-bit QLoRA.

### Limitations

Training loss on selected records is not validation accuracy, balanced class coverage, or proof of a memory benefit from quantization.

### Key Observations

Notebook 06 measures this adapter on the common held-out validation partition.
